# GADES Surface Analysis — Steps 1–3

This notebook implements the first three steps of the GADES post-processing workflow
on the `(x, U, F, H)` data logged during a GADES run.

| Step | What it does |
|------|--------------|
| **1 — Preprocessing** | Projects rigid-body modes out of each Hessian, diagonalises, computes per-snapshot scalar invariants |
| **2 — Feature construction** | Assembles the Z-scored feature matrix Φ ∈ ℝ^{N × (2K+5)} |
| **3 — Clustering** | HDBSCAN on Φ + mode-coherence check; partitions snapshots into landscape regions |

**Required log files** (produced by `GADESBias` when `logfile_prefix` is set):
```
<LOG_PREFIX>_pos.log
<LOG_PREFIX>_epot.log
<LOG_PREFIX>_forces.log
<LOG_PREFIX>_hess.log
```
All four files are expected in the same directory as this notebook.

---
### A note on the subspace Hessian

GADES logs the **subspace Hessian** H_sub — the (3M × 3M) block of the full system
Hessian for the biased atoms only. This differs from the true effective Hessian
(the Schur complement, which marginalises over the bath degrees of freedom):

```
H_eff = H_sub − H_sub,bath · H_bath⁻¹ · H_bath,sub
```

The missing term is positive semi-definite, so **H_sub is systematically softer than
H_eff**. In practice this means:

- Eigenvalues λ_k are underestimated and the **Morse index (number of negative
  eigenvalues) can be non-zero even at true minima**. Do not use the Morse index
  alone to classify snapshots.
- **|F| and f_∥ are unaffected** — they are computed from forces, not from H — and
  are the primary discriminators between near-stationary and active-climbing regions.
- Snapshots closest to true saddles of the full system have small |F|; those where
  GADES is actively climbing have large |f_∥|.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings

import GADES.SurfAnalysis.preprocessing as step1_mod
import GADES.SurfAnalysis.features     as step2_mod
import GADES.SurfAnalysis.clustering   as step3_mod

## Configuration

Edit the variables in this cell to match your run.

In [ ]:
# Path prefix — the logfile_prefix you passed to GADESBias.
# E.g. if your files are named  ./gades_pos.log  set LOG_PREFIX = "gades".
LOG_PREFIX = "gades"

# Simulation temperature in Kelvin (used for quasiharmonic free energy A_i).
TEMPERATURE = 300.0

# Boltzmann constant matching the backend that produced the run:
#   OpenMM  →  8.314462618e-3  kJ mol⁻¹ K⁻¹  (default)
#   ASE     →  8.617333262e-5  eV K⁻¹
KB = 8.314462618e-3

# Atomic masses of the biased atoms (same order as positions in _pos.log).
# Set to None to use uniform masses for the rigid-body projector.
# For covalently bonded subsets embedded in a larger system the rigid-body
# modes are not true zero-frequency modes; set SKIP_RIGID_PROJECTION = True
# in that case (see below).
MASSES = None

# Number of soft-mode eigenvectors to retain from Step 1.
N_EIGVECS = 20

# Number of eigenvalue modes K to include in the feature vector (Step 2).
# None → use all non-rigid modes.
K_FEATURES = None

# HDBSCAN parameters (Step 3).
HDBSCAN_MIN_CLUSTER_SIZE = 20
HDBSCAN_MIN_SAMPLES      = 5
COHERENCE_THRESHOLD      = 0.8

# Force percentile below which a cluster is considered near-stationary.
# Clusters with ⟨|F|⟩ below the FORCE_PERCENTILE-th percentile of all
# per-snapshot forces are flagged as saddle candidates.
FORCE_PERCENTILE = 25

---
## Step 1 — Preprocessing and diagnostics

In [ ]:
s1 = step1_mod.run(
    logfile_prefix = LOG_PREFIX,
    temperature    = TEMPERATURE,
    masses         = MASSES,
    n_eigvecs      = N_EIGVECS,
    kB             = KB,
)

N = len(s1.steps)
print(f"Loaded {N} snapshots  |  n_rigid = {s1.n_rigid}  |  "
      f"n_nontrivial = {s1.eigenvalues.shape[1]}")

if (s1.morse_index >= 1).all():
    print()
    print("NOTE: all snapshots have Morse index ≥ 1.")
    print("This is expected for subspace Hessians (H_sub is softer than H_eff).")
    print("Use |F| and f_∥ — not the Morse index — to identify near-stationary regions.")

### Diagnostic plots

In [ ]:
t = np.arange(N)   # snapshot index

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
fig.suptitle("Step 1 — per-snapshot diagnostics", fontsize=13)

# ── panel 1: energy ──────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(t, s1.U, lw=0.8, label="$U_i$")
ax.plot(t, s1.A, lw=0.8, alpha=0.7, label="$A_i$")
ax.set_ylabel("Energy")
ax.legend(fontsize=9)

# ── panel 2: eigenvalues and spectral gap ────────────────────────────────────
ax = axes[1]
ax.plot(t, s1.eigenvalues[:, 0], lw=0.8, label=r"$\lambda_1$")
if s1.eigenvalues.shape[1] >= 2:
    ax.plot(t, s1.eigenvalues[:, 1], lw=0.8, label=r"$\lambda_2$")
    ax.plot(t, s1.spectral_gap,       lw=0.6, ls="--", label=r"$g = \lambda_2 - \lambda_1$")
ax.axhline(0, color="k", lw=0.5, ls=":")
ax.set_ylabel("Eigenvalue")
ax.legend(fontsize=9)

# ── panel 3: forces ──────────────────────────────────────────────────────────
ax = axes[2]
ax.plot(t, s1.F_norm,     lw=0.8, label="$|F_i|$")
ax.plot(t, s1.f_parallel, lw=0.8, label=r"$f_{\parallel}^{(i)}$")
ax.axhline(0, color="k", lw=0.5, ls=":")
f_thresh = np.percentile(s1.F_norm, FORCE_PERCENTILE)
ax.axhline(f_thresh, color="C3", lw=1, ls="--",
           label=f"$P_{{{FORCE_PERCENTILE}}}(|F|)$ = {f_thresh:.3f}")
ax.set_ylabel("Force")
ax.legend(fontsize=9)

# ── panel 4: Morse index (informational only for subspace Hessians) ──────────
ax = axes[3]
ax.step(t, s1.morse_index, where="mid", lw=0.9, color="C2")
ax.set_ylabel("Morse index $m_i$\n(approx. for H_sub)")
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.set_xlabel("Snapshot index")

for ax in axes:
    ax.grid(True, lw=0.3, alpha=0.5)

plt.tight_layout()
plt.savefig("step1_diagnostics.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Force-based snapshot summary
f_thresh = np.percentile(s1.F_norm, FORCE_PERCENTILE)
n_low_F  = int((s1.F_norm < f_thresh).sum())
print(f"|F| threshold (P{FORCE_PERCENTILE})  : {f_thresh:.4f}")
print(f"Snapshots with |F| < threshold : {n_low_F:4d}  ({100*n_low_F/N:.1f}%)  ← near-stationary")
print(f"NaN A_i (no positive eigenvalues)  : {int(np.isnan(s1.A).sum()):4d}")

---
## Step 2 — Feature construction

In [ ]:
s2 = step2_mod.run(s1, K=K_FEATURES)

print(f"Feature matrix Φ : {s2.Phi.shape}  (K = {s2.K})")
print(f"Feature names     : {s2.feature_names}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Step 2 — feature matrix", fontsize=13)

# Raw feature matrix
im0 = axes[0].imshow(
    s2.Phi_raw.T, aspect="auto", interpolation="none",
    cmap="RdBu_r", origin="lower",
)
axes[0].set_title("Raw  Φ_raw")
axes[0].set_xlabel("Snapshot index")
axes[0].set_ylabel("Feature")
axes[0].set_yticks(range(len(s2.feature_names)))
axes[0].set_yticklabels(s2.feature_names, fontsize=7)
plt.colorbar(im0, ax=axes[0])

# Standardised feature matrix
vmax = np.percentile(np.abs(s2.Phi), 98)
im1 = axes[1].imshow(
    s2.Phi.T, aspect="auto", interpolation="none",
    cmap="RdBu_r", vmin=-vmax, vmax=vmax, origin="lower",
)
axes[1].set_title("Standardised  Φ")
axes[1].set_xlabel("Snapshot index")
axes[1].set_yticks(range(len(s2.feature_names)))
axes[1].set_yticklabels(s2.feature_names, fontsize=7)
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.savefig("step2_features.pdf", bbox_inches="tight")
plt.show()

---
## Step 3 — Clustering into landscape regions

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    s3 = step3_mod.run(
        s2, s1,
        min_cluster_size    = HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples         = HDBSCAN_MIN_SAMPLES,
        coherence_threshold = COHERENCE_THRESHOLD,
    )

for w in caught:
    print(f"[warning] {w.message}")

n_clusters = len(s3.cluster_ids)
print(f"Clusters found : {n_clusters}")
print(f"Noise points   : {s3.n_noise} / {N}  ({100*s3.n_noise/N:.1f}%)")

In [ ]:
# Pre-compute per-cluster force statistics (used throughout this section)
mean_F  = {int(cid): s1.F_norm[s3.labels == cid].mean()    for cid in s3.cluster_ids}
mean_fp = {int(cid): s1.f_parallel[s3.labels == cid].mean() for cid in s3.cluster_ids}

# Force threshold for near-stationary classification
f_thresh = np.percentile(s1.F_norm, FORCE_PERCENTILE)

In [ ]:
# Per-cluster summary table
# Classification uses |F|, not Morse index (see note at top of notebook).
print(f"{'Cluster':>7}  {'Size':>5}  {'⟨U⟩':>10}  {'⟨λ₁⟩':>10}  "
      f"{'⟨|F|⟩':>9}  {'⟨f∥⟩':>8}  {'sC':>5}  Region")
print("-" * 75)
for j, cid in enumerate(s3.cluster_ids):
    mF   = mean_F[int(cid)]
    mfp  = mean_fp[int(cid)]
    near = mF < f_thresh
    region = "near-stationary" if near else "active"
    print(
        f"{cid:>7d}  "
        f"{s3.cluster_sizes[j]:>5d}  "
        f"{s3.cluster_mean_U[j]:>10.3f}  "
        f"{s3.cluster_mean_lambda1[j]:>10.3f}  "
        f"{mF:>9.4f}  "
        f"{mfp:>8.4f}  "
        f"{s3.cluster_coherence[j]:>5.2f}  "
        f"{region}"
    )

In [ ]:
# Build colour map and legend handles
unique_labels = np.unique(s3.labels)
cmap_colors   = plt.get_cmap("tab20", max(n_clusters, 1))

label_colors = {}
cluster_idx  = 0
for lbl in sorted(unique_labels):
    if lbl == -1:
        label_colors[lbl] = (0.7, 0.7, 0.7, 1.0)
    else:
        label_colors[lbl] = cmap_colors(cluster_idx)
        cluster_idx += 1

point_colors = np.array([label_colors[l] for l in s3.labels])

from matplotlib.patches import Patch
handles = [Patch(color=label_colors[-1], label="noise (-1)")]
for j, cid in enumerate(s3.cluster_ids):
    mF     = mean_F[int(cid)]
    region = "near-stat." if mF < f_thresh else "active"
    handles.append(Patch(
        color=label_colors[int(cid)],
        label=f"C{cid} ({region}, n={s3.cluster_sizes[j]})"
    ))

In [ ]:
# Cluster assignment over time, coloured by cluster
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
fig.suptitle("Step 3 — cluster assignment", fontsize=13)

axes[0].scatter(t, s3.labels, c=point_colors, s=4, linewidths=0)
axes[0].set_ylabel("Cluster label")
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].legend(handles=handles, fontsize=7, loc="upper right", ncol=2)

axes[1].scatter(t, s1.U, c=point_colors, s=4, linewidths=0)
axes[1].set_ylabel("$U_i$")

axes[2].scatter(t, s1.F_norm, c=point_colors, s=4, linewidths=0)
axes[2].axhline(f_thresh, color="k", lw=1, ls="--",
                label=f"$P_{{{FORCE_PERCENTILE}}}$ threshold")
axes[2].set_ylabel("$|F_i|$")
axes[2].set_xlabel("Snapshot index")
axes[2].legend(fontsize=8)

for ax in axes:
    ax.grid(True, lw=0.3, alpha=0.5)

plt.tight_layout()
plt.savefig("step3_clusters.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# 2-D PCA of Φ coloured by cluster label
from numpy.linalg import svd

Phi_c    = s2.Phi - s2.Phi.mean(axis=0)
_, _, Vt = svd(Phi_c, full_matrices=False)
PC       = Phi_c @ Vt[:2].T

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(PC[:, 0], PC[:, 1], c=point_colors, s=8, linewidths=0, alpha=0.7)

for j, cid in enumerate(s3.cluster_ids):
    mask = s3.labels == cid
    cx, cy = PC[mask, 0].mean(), PC[mask, 1].mean()
    ax.text(cx, cy, f"C{cid}", ha="center", va="center", fontsize=8,
            color="black", fontweight="bold")

ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("PCA of Φ — coloured by cluster")
ax.grid(True, lw=0.3, alpha=0.5)
ax.legend(handles=handles, fontsize=7, loc="best", ncol=2)

plt.tight_layout()
plt.savefig("step3_pca.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Mode-coherence bar chart
if n_clusters > 0:
    fig, ax = plt.subplots(figsize=(max(4, n_clusters * 0.6), 3))
    ax.bar(
        range(n_clusters), s3.cluster_coherence,
        color=[label_colors[int(cid)] for cid in s3.cluster_ids],
        edgecolor="k", linewidth=0.5,
    )
    ax.axhline(COHERENCE_THRESHOLD, color="k", ls="--", lw=1,
               label=f"threshold = {COHERENCE_THRESHOLD}")
    ax.set_xticks(range(n_clusters))
    ax.set_xticklabels([f"C{cid}" for cid in s3.cluster_ids])
    ax.set_ylabel(r"Coherence score $s_C$")
    ax.set_ylim(0, 1.05)
    ax.set_title("Mode coherence per cluster")
    ax.legend()
    ax.grid(True, axis="y", lw=0.3, alpha=0.5)
    plt.tight_layout()
    plt.savefig("step3_coherence.pdf", bbox_inches="tight")
    plt.show()

---
## Saddle candidates

All clusters ranked by ⟨|F|⟩ ascending. Clusters near the top have the smallest
forces and are closest to stationary points of the full system — the best seeds for
TS refinement (Step 4).

The Morse index ⟨m⟩ is shown for reference but should not be used as the primary
discriminator when all snapshots come from a subspace Hessian (see note at top).

In [ ]:
if n_clusters == 0:
    print("No clusters found — all snapshots labelled as noise.")
else:
    print(f"{'Cluster':>7}  {'Size':>5}  {'⟨U⟩':>10}  {'⟨λ₁⟩':>10}  "
          f"{'⟨|F|⟩':>9}  {'⟨m⟩':>5}  {'sC':>5}")
    print("-" * 65)
    for cid in sorted(s3.cluster_ids, key=lambda c: mean_F[int(c)]):
        j = np.where(s3.cluster_ids == cid)[0][0]
        marker = " ←" if mean_F[int(cid)] < f_thresh else ""
        print(
            f"{cid:>7d}  "
            f"{s3.cluster_sizes[j]:>5d}  "
            f"{s3.cluster_mean_U[j]:>10.3f}  "
            f"{s3.cluster_mean_lambda1[j]:>10.3f}  "
            f"{mean_F[int(cid)]:>9.4f}  "
            f"{s3.cluster_mean_morse[j]:>5.2f}  "
            f"{s3.cluster_coherence[j]:>5.2f}"
            f"{marker}"
        )
    print(f"\n← near-stationary (⟨|F|⟩ < P{FORCE_PERCENTILE} = {f_thresh:.4f})")

In [ ]:
# Scatter plot: ⟨|F|⟩ vs ⟨λ₁⟩ per cluster — the two primary landscape descriptors
if n_clusters > 0:
    fig, ax = plt.subplots(figsize=(6, 5))

    for j, cid in enumerate(s3.cluster_ids):
        mF = mean_F[int(cid)]
        ml = s3.cluster_mean_lambda1[j]
        ax.scatter(mF, ml, s=s3.cluster_sizes[j] * 0.5,
                   color=label_colors[int(cid)], edgecolors="k", linewidths=0.5,
                   zorder=3)
        ax.annotate(f"C{cid}", (mF, ml), textcoords="offset points",
                    xytext=(4, 4), fontsize=8)

    ax.axvline(f_thresh, color="k", lw=1, ls="--",
               label=f"$P_{{{FORCE_PERCENTILE}}}$ threshold")
    ax.axhline(0, color="gray", lw=0.5, ls=":")
    ax.set_xlabel(r"$\langle |F| \rangle$  (cluster mean force)")
    ax.set_ylabel(r"$\langle \lambda_1 \rangle$  (cluster mean softest eigenvalue)")
    ax.set_title("Landscape map: force vs. curvature per cluster\n"
                 "(marker size ∝ cluster size)")
    ax.legend(fontsize=8)
    ax.grid(True, lw=0.3, alpha=0.5)

    plt.tight_layout()
    plt.savefig("step3_landscape_map.pdf", bbox_inches="tight")
    plt.show()

---
## Export results

Save the step results to NumPy files for downstream use (Steps 4–6).

In [ ]:
np.save("eigenvalues.npy",  s1.eigenvalues)
np.save("eigenvectors.npy", s1.eigenvectors)
np.save("features.npy",     s2.Phi)
np.save("labels.npy",       s3.labels)

import csv
with open("cluster_summary.csv", "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["cluster_id", "size", "mean_U", "mean_lambda1",
                     "mean_morse", "mean_F", "mean_f_parallel", "coherence"])
    for j, cid in enumerate(s3.cluster_ids):
        writer.writerow([
            int(cid),
            int(s3.cluster_sizes[j]),
            float(s3.cluster_mean_U[j]),
            float(s3.cluster_mean_lambda1[j]),
            float(s3.cluster_mean_morse[j]),
            float(mean_F[int(cid)]),
            float(mean_fp[int(cid)]),
            float(s3.cluster_coherence[j]),
        ])

print("Saved: eigenvalues.npy, eigenvectors.npy, features.npy, labels.npy, cluster_summary.csv")